In [1]:
from google.colab import files
uploaded = files.upload()

Saving archive.zip to archive.zip


In [2]:
import zipfile

zip_path = list(uploaded.keys())[0]  # auto gets file name
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted successfully!")

Extracted successfully!


In [3]:
import os
print(os.listdir("/content/dataset"))

['Dataset_BUSI_with_GT']


In [4]:
data_dir = "/content/dataset/Dataset_BUSI_with_GT"

In [5]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_data = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

Found 1578 files belonging to 3 classes.
Using 1263 files for training.
Found 1578 files belonging to 3 classes.
Using 315 files for validation.


In [6]:
class_names = train_data.class_names
print(class_names)

['benign', 'malignant', 'normal']


STEP 4: Build the Model (Using MobileNetV2 — best choice)**bold text**

---



In [7]:
from tensorflow.keras import layers, models

In [8]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(3, activation='softmax')  # 3 classes
])

In [10]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

STEP 5: Train the Model


---



---



In [12]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 49s 777ms/step - accuracy: 0.7118 - loss: 0.6711 - val_accuracy: 0.8063 - val_loss: 0.4571
Epoch 2/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 146ms/step - accuracy: 0.8203 - loss: 0.4335 - val_accuracy: 0.8190 - val_loss: 0.4293
Epoch 3/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 128ms/step - accuracy: 0.8345 - loss: 0.3909 - val_accuracy: 0.8476 - val_loss: 0.3960
Epoch 4/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 126ms/step - accuracy: 0.8488 - loss: 0.3627 - val_accuracy: 0.8317 - val_loss: 0.3863
Epoch 5/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 136ms/step - accuracy: 0.8757 - loss: 0.3239 - val_accuracy: 0.8444 - val_loss: 0.3982


STEP 6: Convert Model to TensorFlow Lite


---




In [13]:
model.save("breast_cancer_model.h5")

In [14]:
import tensorflow as tf

# Load model
model = tf.keras.models.load_model("breast_cancer_model.h5")

# Convert
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save file
with open("breast_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model created!")

Saved artifact at '/tmp/tmptu9bs1xy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  135338450704848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135338450706000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135338450706768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135338450698320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135338450707536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135338450706960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135338450706576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135338450707344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135338450707152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135338450708496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1353384507079

sampling


---



---



In [5]:
import os
print(os.listdir())

['.config', 'breast_cancer_model.tflite', 'sample_data']


In [6]:
import tensorflow as tf
import numpy as np
import cv2

interpreter = tf.lite.Interpreter(model_path="breast_cancer_model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(input_details)
print(output_details)

[{'name': 'serving_default_input_layer_1:0', 'index': 0, 'shape': array([  1, 224, 224,   3], dtype=int32), 'shape_signature': array([ -1, 224, 224,   3], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
[{'name': 'StatefulPartitionedCall_1:0', 'index': 175, 'shape': array([1, 3], dtype=int32), 'shape_signature': array([-1,  3], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


In [7]:
print(input_details)


[{'name': 'serving_default_input_layer_1:0', 'index': 0, 'shape': array([  1, 224, 224,   3], dtype=int32), 'shape_signature': array([ -1, 224, 224,   3], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


In [8]:
from google.colab import files
files.upload()

Saving benign (1).png to benign (1).png


{'benign (1).png': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x022\x00\x00\x01\xd7\x08\x02\x00\x00\x00)\xcf\'\xa2\x00\x00 \x00IDATx\x01d\xc1M\x08\xaemU\x06\xec\xf3\\k]\xf7\xfd\xecW\xc5(\x07\xd1\xb0I\xc3\x08\xfa\x83\x10"i\x16F\x83\x08\x13,\x94p\xd2\x1f\x04\xe1\xa4\x81M\x8a\x06!HN*\x82h\xe4\xa4A\x83&j!\xa5F\xafh)M\xb2 )\xc8"\xb4\xcfw\xef\xe7\xb9\xafk\xadu~\x1f\x17<\xb0\xe3;\x0e\x9e\xe7\xb9\xd623w\xbf\xae\xeb\xc5\x8b\x17\x99\t`\xad\x15\x11U\x15\x11k\xad\xb7\xbf\xfd\xed\x9f\xfb\xdc\xe7\xae\xeb\x92TU\xddM\xd2\xcc"\xe2\xd8\xba{\xad\xf5x<\xae\xeb\xca\xcc\xee\xe6\xe6\xee$\x01TUn\xdd\r\x80$\xb6\x88\x18c\xb8;I\x00\xd7uuwfV\x15Iw\')\t[D\xb8;\x80\xcc\xac*\x00$\xbb\x1b\x00\xc9\xb1E\x84\xbbK\x02\xc0\r\x80\xa4\xee\x96TU\x00\xcc\xcc\x9fHJ\x02@\x12\xaf\xe9\xee\xaa\x02 \xa97\x92\x11af\x00\xaa\xaa\xbb\xabJ\xd2\xd8HV\x95\xa4\xb9u\xb7\xa4\xcc\xecnI\x99))"^\xbcx\xf1\xc6\x1bo\x8c1$uwU\xf15\x00\xba[RU\x91\xec\xee\xcc\xac*I\xb5\xb9\xbb\x99IZku76I\xdd\x8d\x8ddD\x8c1\xccLRo$\xdd\xdd\xccj\x93D\x12\x80\x99\x8d1

In [10]:
import os
print(os.listdir())

['.config', 'benign (1).png', 'breast_cancer_model.tflite', 'sample_data']


In [11]:
import cv2
import numpy as np

IMG_SIZE = 224

img = cv2.imread("benign (1).png")

if img is None:
    print("❌ Image not loaded")
else:
    print("✅ Image loaded")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img.astype(np.float32), axis=0)

    print("Processed shape:", img.shape)

✅ Image loaded
Processed shape: (1, 224, 224, 3)


In [12]:
interpreter.set_tensor(input_details[0]['index'], img)
interpreter.invoke()

output = interpreter.get_tensor(output_details[0]['index'])
print("Raw Output:", output)

Raw Output: [[9.9918729e-01 8.0111640e-04 1.1608956e-05]]


Convert to prediction


In [13]:
import numpy as np

pred = np.argmax(output)
print("Predicted class index:", pred)

Predicted class index: 0


In [14]:
class_names = ["Benign", "Malignant", "Normal"]  # ⚠️ adjust this!

print("Prediction:", class_names[pred])

Prediction: Benign
